# NegConstraint: download and inspect the released dataset

This notebook loads the public NegConstraint release accompanying [Logical Consistency is Vital](https://aclanthology.org/2025.findings-acl.92/). It uses only Python's standard library and runs directly in Google Colab.

NegConstraint contains retrieval queries with explicit negative constraints. The dataset is stored in a BEIR-like corpus/query/qrels layout.

In [ ]:
import csv
import io
import json
import urllib.request
import zipfile

DATA_URL = "https://raw.githubusercontent.com/xgl-git/NS-IR-main/main/NegConstraint.zip"

with urllib.request.urlopen(DATA_URL) as response:
    archive_bytes = response.read()

print(f"Downloaded {len(archive_bytes) / 1_000_000:.2f} MB")
archive = zipfile.ZipFile(io.BytesIO(archive_bytes))
archive.namelist()

## Load the corpus, queries, and relevance judgments

The files are read directly from memory; the ZIP does not need to be extracted into the Colab filesystem.

In [ ]:
def read_jsonl_from_zip(archive, member):
    with archive.open(member) as raw:
        text = io.TextIOWrapper(raw, encoding="utf-8")
        return [json.loads(line) for line in text if line.strip()]

def read_tsv_from_zip(archive, member):
    with archive.open(member) as raw:
        text = io.TextIOWrapper(raw, encoding="utf-8")
        return list(csv.DictReader(text, delimiter="\t"))

corpus_rows = read_jsonl_from_zip(archive, "NegConstraint/corpus.jsonl")
query_rows = read_jsonl_from_zip(archive, "NegConstraint/queries.jsonl")
qrel_rows = read_tsv_from_zip(archive, "NegConstraint/test.tsv")

corpus = {row["_id"]: row for row in corpus_rows}
queries = {row["_id"]: row for row in query_rows}

print("Documents:", len(corpus))
print("Queries:  ", len(queries))
print("Qrels:    ", len(qrel_rows))

## Validate the release

Every qrel should point to an existing query and document. We also compare the released query IDs with those that have relevance judgments.

In [ ]:
qrel_query_ids = {row["query-id"] for row in qrel_rows}
qrel_corpus_ids = {row["corpus-id"] for row in qrel_rows}

assert qrel_query_ids <= set(queries), "A qrel references an unknown query."
assert qrel_corpus_ids <= set(corpus), "A qrel references an unknown document."
assert all(int(row["score"]) == 1 for row in qrel_rows)

missing_qrels = sorted(set(queries) - qrel_query_ids, key=int)
print(f"Judged queries:   {len(qrel_query_ids)} / {len(queries)}")
print(f"Unjudged queries: {len(missing_qrels)}")
print("First missing query IDs:", missing_qrels[:10])

# Values observed in the public archive checked for this notebook.
assert len(corpus) == 3946
assert len(queries) == 366
assert len(qrel_rows) == 198
assert missing_qrels[0] == "198"

## Join two queries to their positive documents

The released qrels contain one positive document per judged query. Text is truncated only for display.

In [ ]:
examples = []
for qrel in qrel_rows[:2]:
    query = queries[qrel["query-id"]]
    document = corpus[qrel["corpus-id"]]
    examples.append({
        "query": query,
        "positive_document": document,
        "relevance": int(qrel["score"]),
    })

for example in examples:
    print("QUERY", example["query"]["_id"], "—", example["query"]["text"])
    text = example["positive_document"]["text"]
    print("POSITIVE DOC", example["positive_document"]["_id"], "—", text[:300] + "…")
    print()

## Evaluation warning

Use only the 198 query IDs present in `test.tsv` for evaluation unless you obtain labels for the other 168 queries. An absent relevance judgment means **unjudged**, not irrelevant. The authors' repository does not currently state an explicit dataset license, so confirm reuse terms before redistribution or commercial use.